In [ ]:
  Design Notification system

    Plug and play model
    Extendible : sms,email
    Notification : Dynamically
    Store all notification / logging

In [ ]:
  <abstract>
INotification

  getContent()                               INotificationDecorator
                                                Inotification not()


SimpleNotification
  getContent()


 INotificationDecorator
      getContent() 

SignatureDecorator
     getContent() 

In [1]:
from abc import ABC, abstractmethod
from datetime import datetime

"""
=========================================
        DECORATOR PATTERN
=========================================
"""

# Component Interface
class INotification(ABC):
    @abstractmethod
    def get_content(self):
        pass


# Concrete Component
class SimpleNotification(INotification):
    def __init__(self, msg):
        self.text = msg

    def get_content(self):
        return self.text


# Abstract Decorator
class INotificationDecorator(INotification):
    def __init__(self, notification: INotification):
        self.notification = notification


# Concrete Decorator – Add Timestamp
class TimestampDecorator(INotificationDecorator):
    def get_content(self):
        ts = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
        return ts + self.notification.get_content()


# Concrete Decorator – Add Signature
class SignatureDecorator(INotificationDecorator):
    def __init__(self, notification: INotification, signature: str):
        super().__init__(notification)
        self.signature = signature

    def get_content(self):
        return self.notification.get_content() + f"\n-- {self.signature}\n"


"""
=========================================
         OBSERVER PATTERN
=========================================
"""

class IObserver(ABC):
    @abstractmethod
    def update(self):
        pass


class IObservable(ABC):
    @abstractmethod
    def add_observer(self, observer: "IObserver"):
        pass

    @abstractmethod
    def remove_observer(self, observer: "IObserver"):
        pass

    @abstractmethod
    def notify_observers(self):
        pass


# Concrete Observable
class NotificationObservable(IObservable):
    def __init__(self):
        self.observers = []
        self.current_notification: INotification = None

    def add_observer(self, observer: IObserver):
        self.observers.append(observer)

    def remove_observer(self, observer: IObserver):
        self.observers.remove(observer)

    def notify_observers(self):
        for obs in self.observers:
            obs.update()

    def set_notification(self, notification: INotification):
        self.current_notification = notification
        self.notify_observers()

    def get_notification_content(self):
        return self.current_notification.get_content()


"""
=========================================
      SINGLETON + OBSERVABLE
=========================================
"""

class NotificationService:
    """
    Singleton Pattern + Uses Observable
    """
    _instance = None

    def __init__(self):
        self.observable = NotificationObservable()
        self.notifications = []

    @staticmethod
    def get_instance():
        if NotificationService._instance is None:
            NotificationService._instance = NotificationService()
        return NotificationService._instance

    def send_notification(self, notification: INotification):
        self.notifications.append(notification)
        self.observable.set_notification(notification)


"""
=========================================
         OBSERVER #1 – Logger
=========================================
"""

class Logger(IObserver):
    def __init__(self, observable=None):
        if observable is None:
            observable = NotificationService.get_instance().observable

        self.observable = observable
        self.observable.add_observer(self)

    def update(self):
        print("📝 Logging Notification:\n", self.observable.get_notification_content())


"""
=========================================
   STRATEGY PATTERN (Used by Observer #2)
=========================================
"""

class INotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, content):
        pass


class EmailStrategy(INotificationStrategy):
    def __init__(self, email_id):
        self.email_id = email_id

    def send_notification(self, content):
        print(f"📧 Sending Email to {self.email_id}\n{content}")


class SMSStrategy(INotificationStrategy):
    def __init__(self, number):
        self.number = number

    def send_notification(self, content):
        print(f"📩 Sending SMS to {self.number}\n{content}")


class PopUpStrategy(INotificationStrategy):
    def send_notification(self, content):
        print(f"🔔 Popup Notification:\n{content}")


"""
=========================================
   OBSERVER #2 – Notification Engine
   (Uses Strategy Pattern)
=========================================
"""

class NotificationEngine(IObserver):
    def __init__(self, observable=None):
        if observable is None:
            observable = NotificationService.get_instance().observable

        self.observable = observable
        self.observable.add_observer(self)

        self.strategies = []

    def add_notification_strategy(self, strategy: INotificationStrategy):
        self.strategies.append(strategy)

    def update(self):
        notification_content = self.observable.get_notification_content()

        # Send through all attached strategies
        for strategy in self.strategies:
            strategy.send_notification(notification_content)


"""
=========================================
                MAIN
=========================================
"""

if __name__ == "__main__":

    # Singleton Service
    notification_service = NotificationService.get_instance()

    # Observer 1
    logger = Logger()

    # Observer 2
    engine = NotificationEngine()
    engine.add_notification_strategy(EmailStrategy("random.person@gmail.com"))
    engine.add_notification_strategy(SMSStrategy("+91 9876543210"))
    engine.add_notification_strategy(PopUpStrategy())

    # Base Notification
    notification = SimpleNotification("Your order has been shipped!")

    # Decorators applied dynamically
    notification = TimestampDecorator(notification)
    notification = SignatureDecorator(notification, "Customer Care")

    # Notify all observers
    notification_service.send_notification(notification)


📝 Logging Notification:
 [2025-12-26 14:31:51] Your order has been shipped!
-- Customer Care

📧 Sending Email to random.person@gmail.com
[2025-12-26 14:31:51] Your order has been shipped!
-- Customer Care

📩 Sending SMS to +91 9876543210
[2025-12-26 14:31:51] Your order has been shipped!
-- Customer Care

🔔 Popup Notification:
[2025-12-26 14:31:51] Your order has been shipped!
-- Customer Care



In [2]:
from abc import ABC, abstractmethod
from datetime import datetime
# ============================
# DECORATOR PATTERN
# ============================

# Base interface
class INotification(ABC):
    @abstractmethod
    def get_content(self):
        pass

# Concrete Component
class SimpleNotification(INotification):
    def __init__(self, msg):
        self.text = msg

    def get_content(self):
        return self.text

# Abstract Decorator
class INotificationDecorator(INotification):
    def __init__(self, notification: INotification):
        self.notification = notification

# Concrete Decorator - Timestamp
class TimestampDecorator(INotificationDecorator):
    def get_content(self):
        ts = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
        return ts + self.notification.get_content()

# Concrete Decorator - Signature
class SignatureDecorator(INotificationDecorator):
    def __init__(self, notification, signature):
        super().__init__(notification)
        self.signature = signature

    def get_content(self):
        return self.notification.get_content() + f"\n-- {self.signature}"

# ============================
# RUN
# ============================
if __name__ == "__main__":
    n = SimpleNotification("Your order has been shipped!")
    n = TimestampDecorator(n)       # add timestamp dynamically
    n = SignatureDecorator(n, "Customer Service")  # add signature dynamically

    print("\nFinal Message:")
    print(n.get_content())



Final Message:
[2025-12-26 14:34:51] Your order has been shipped!
-- Customer Service


In [24]:
# Abstract Decorator exampe

from abc import ABC,abstractmethod

class Cofee(ABC):
    @abstractmethod
    def cost(self):
        pass

class SimpleCofee(Cofee):
    def cost(self):
        return 50

# 3️⃣ ABSTRACT DECORATOR
class CoffeeDecorator(Cofee):
    def __init__(self,cofee:Cofee):
        self.cofee = cofee

# 4️⃣ Concrete Decorators
# Milk Add-on
class MilkDecorator(CoffeeDecorator):
    def cost(self):
        return self.cofee.cost() + 10

class ChocolateDecorator(CoffeeDecorator):
    def cost(self):
        return self.cofee.cost() + 20

In [25]:
coffee = SimpleCofee()

coffee = MilkDecorator(coffee)
coffee = ChocolateDecorator(coffee)

print(coffee.cost())
print(coffee.cost())

80
80


In [26]:
from abc import ABC, abstractmethod
from datetime import datetime

# ============================
# DECORATOR PATTERN
# ============================

# Base interface
class INotification(ABC):
    @abstractmethod
    def get_content(self):
        pass

# Concrete Component
class SimpleNotification(INotification):
    def __init__(self, msg):
        self.text = msg

    def get_content(self):
        return self.text

# Abstract Decorator
class INotificationDecorator(INotification):
    def __init__(self, notification: INotification):
        self.notification = notification

# Concrete Decorator - Timestamp
class TimestampDecorator(INotificationDecorator):
    def get_content(self):
        ts = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
        return ts + self.notification.get_content()

# Concrete Decorator - Signature
class SignatureDecorator(INotificationDecorator):
    def __init__(self, notification, signature):
        super().__init__(notification)
        self.signature = signature

    def get_content(self):
        return self.notification.get_content() + f"\n-- {self.signature}"

# ============================
# RUN
# ============================
if __name__ == "__main__":
    n = SimpleNotification("Your order has been shipped!")
    n = TimestampDecorator(n)       # add timestamp dynamically
    n = SignatureDecorator(n, "Customer Service")  # add signature dynamically

    print("\nFinal Message:")
    print(n.get_content())



Final Message:
[2025-12-26 15:29:29] Your order has been shipped!
-- Customer Service


In [ ]:
✅ SUBPART–2: OBSERVER PATTERN (Runnable Code)

In [27]:
from abc import ABC, abstractmethod

# ============================================
# OBSERVER PATTERN
# ============================================

# Observer Interface
class IObserver(ABC):
    @abstractmethod
    def update(self, message):
        pass

# Observable Interface
class IObservable(ABC):
    @abstractmethod
    def add_observer(self, observer):
        pass

    @abstractmethod
    def remove_observer(self, observer):
        pass

    @abstractmethod
    def notify_observers(self, message):
        pass


# Concrete Observable
class NotificationObservable(IObservable):
    def __init__(self):
        self.observers = []

    def add_observer(self, observer: IObserver):
        self.observers.append(observer)

    def remove_observer(self, observer: IObserver):
        self.observers.remove(observer)

    def notify_observers(self, message):
        for obs in self.observers:
            obs.update(message)

    def new_notification(self, message):
        print("\n📢 New Notification Triggered!")
        self.notify_observers(message)


# Concrete Observer
class Logger(IObserver):
    def update(self, message):
        print(f"📝 Logger Received Notification:\n{message}")


class PopupNotifier(IObserver):
    def update(self, message):
        print(f"🔔 Popup Shown:\n{message}")


# ============================================
# RUN — DEMO
# ============================================
if __name__ == "__main__":
    # Create observable
    observable = NotificationObservable()

    # Create observers
    logger = Logger()
    popup = PopupNotifier()

    # Attach observers
    observable.add_observer(logger)
    observable.add_observer(popup)

    # Send a notification
    observable.new_notification("Your order has been packed!")

    # Remove popup observer
    observable.remove_observer(popup)

    print("\n📢 Sending second notification...")
    observable.new_notification("Your order is out for delivery!")



📢 New Notification Triggered!
📝 Logger Received Notification:
Your order has been packed!
🔔 Popup Shown:
Your order has been packed!

📢 Sending second notification...

📢 New Notification Triggered!
📝 Logger Received Notification:
Your order is out for delivery!


In [ ]:
✅ SUBPART–3: STRATEGY PATTERN (Runnable Code)

In [28]:
from abc import ABC, abstractmethod

# ============================================
# STRATEGY PATTERN
# ============================================

# Strategy Interface
class INotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message):
        pass


# Concrete Strategy - Email
class EmailStrategy(INotificationStrategy):
    def __init__(self, email):
        self.email = email

    def send_notification(self, message):
        print(f"📧 Email sent to {self.email}:\n{message}")


# Concrete Strategy - SMS
class SMSStrategy(INotificationStrategy):
    def __init__(self, mobile):
        self.mobile = mobile

    def send_notification(self, message):
        print(f"📩 SMS sent to {self.mobile}:\n{message}")


# Concrete Strategy - Popup
class PopupStrategy(INotificationStrategy):
    def send_notification(self, message):
        print(f"🔔 Popup message:\n{message}")


# Context class
class NotificationSender:
    def __init__(self, strategy: INotificationStrategy):
        self.strategy = strategy  # chosen strategy

    def set_strategy(self, strategy: INotificationStrategy):
        self.strategy = strategy

    def send(self, message):
        self.strategy.send_notification(message)


# ============================================
# RUN — DEMO
# ============================================
if __name__ == "__main__":
    # Set initial strategy as Email
    sender = NotificationSender(EmailStrategy("user@example.com"))
    sender.send("Your order has been confirmed!")

    # Change to SMS dynamically
    sender.set_strategy(SMSStrategy("+91 9876543210"))
    sender.send("Your order has been shipped!")

    # Change to Popup dynamically
    sender.set_strategy(PopupStrategy())
    sender.send("Your package is out for delivery!")


📧 Email sent to user@example.com:
Your order has been confirmed!
📩 SMS sent to +91 9876543210:
Your order has been shipped!
🔔 Popup message:
Your package is out for delivery!
